# Economic Growth Rankings & Monte-Carlo Simulation — Solution
## R Vectors, Lists, Functions, Apply Family + Economic Simulation

Complete solution with alternates, extra practice, and a fully parameterised Monte-Carlo engine for growth rankings under uncertainty.

![Flowchart](economic_growth_rankings_flowchart.png)

![Simulation plots](economic_growth_simulation_plots.png)


## 0. Reference data

In [ ]:
growth_ranking <- c("Ireland", "India", "Vietnam", "CostaRica", "Chile",
                    "Poland", "Mexico", "Brazil", "SouthAfrica", "Argentina")

info_list <- list(
  Ireland = list(growth = 5.2, inflation = 2.1, region = "Europe", debt = 52),
  India   = list(growth = 6.8, inflation = 4.5, region = "Asia",   debt = 82)
)
print(growth_ranking)


## 1. countries vector

In [ ]:
countries <- c("CostaRica", "Chile", "Mexico")
print(countries)


## 2. Extend info_list

In [ ]:
info_list[["CostaRica"]] <- list(growth = 4.1, inflation = 3.2, region = "LatAm", debt = 68)
info_list[["Chile"]]     <- list(growth = 2.4, inflation = 3.8, region = "LatAm", debt = 38)
info_list[["Mexico"]]    <- list(growth = 2.0, inflation = 4.7, region = "LatAm", debt = 49)

str(info_list[c("CostaRica", "Chile", "Mexico")])


## 3. print_country_profile (with loop)

In [ ]:
print_country_profile <- function(name) {
  info <- info_list[[name]]
  print(paste0(name, ": growth=", info$growth, "%, inflation=", info$inflation,
               "%, region=", info$region, ", debt=", info$debt, "% of GDP"))
}

for (c in countries) print_country_profile(c)


## 4–10. find_rank + tests

In [ ]:
find_rank <- function(country) {
  for (rank in 1:length(growth_ranking)) {
    if (growth_ranking[rank] == country) return(rank)
  }
  return(length(growth_ranking) + 1)
}

cat("CostaRica ->", find_rank("CostaRica"), "(expected 4)\n")
cat("Chile     ->", find_rank("Chile"),     "(expected 5)\n")
cat("Mexico    ->", find_rank("Mexico"),    "(expected 7)\n")
cat("Peru      ->", find_rank("Peru"),      "(expected 11)\n")


## 11–12. lapply / sapply

In [ ]:
cat("lapply:\n"); print(lapply(countries, find_rank))
cat("\nsapply:\n"); print(sapply(countries, find_rank))
# Expected sapply: CostaRica=4, Chile=5, Mexico=7


## Alternate Implementations

In [ ]:
find_rank_match <- function(country) {
  idx <- match(country, growth_ranking)
  if (is.na(idx)) length(growth_ranking) + 1L else as.integer(idx)
}

find_rank_which <- function(country) {
  idx <- which(growth_ranking == country)
  if (length(idx) == 0) length(growth_ranking) + 1L else idx[1]
}

# Named lookup
rank_lookup <- setNames(seq_along(growth_ranking), growth_ranking)
find_rank_named <- function(country) {
  p <- rank_lookup[[country]]
  if (is.null(p)) length(growth_ranking) + 1L else p
}

test <- c(countries, "Peru")
data.frame(
  country = test,
  loop  = sapply(test, find_rank),
  match = sapply(test, find_rank_match),
  which = sapply(test, find_rank_which),
  named = sapply(test, find_rank_named)
)


## More Practice — Solutions

In [ ]:
# 1. Countries in a region
countries_in_region <- function(region) {
  countries[sapply(countries, function(c) info_list[[c]]$region == region)]
}
cat("LatAm focus economies:", countries_in_region("LatAm"), "\n")

# 2. Data frame
ranks <- sapply(countries, find_rank)
econ_df <- data.frame(
  country   = countries,
  growth    = sapply(countries, function(c) info_list[[c]]$growth),
  inflation = sapply(countries, function(c) info_list[[c]]$inflation),
  debt      = sapply(countries, function(c) info_list[[c]]$debt),
  rank      = ranks,
  stringsAsFactors = FALSE
)
print(econ_df)

# 3. Best (lowest) rank
best <- econ_df$country[which.min(econ_df$rank)]
cat("\nBest-ranked focus economy:", best, "at rank", min(econ_df$rank), "\n")


## Simulation Section — Full Monte-Carlo Engine
We draw growth rates from independent normal distributions centred on each country’s baseline, then re-rank in every replication.


In [ ]:
set.seed(42)
n_sims   <- 2000
baseline <- c(CostaRica = 4.1, Chile = 2.4, Mexico = 2.0)
sigma    <- c(CostaRica = 1.3, Chile = 1.0, Mexico = 1.5)

# Growth matrix (n_sims × 3)
growth_mat <- matrix(rnorm(n_sims * 3, mean = rep(baseline, each = n_sims),
                           sd = rep(sigma, each = n_sims)),
                     nrow = n_sims, ncol = 3)
colnames(growth_mat) <- names(baseline)

# Rank matrix: 1 = highest growth in that simulation
rank_mat <- t(apply(growth_mat, 1, function(row) rank(-row, ties.method = "min")))
colnames(rank_mat) <- names(baseline)

cat("Mean ranks:\n"); print(round(colMeans(rank_mat), 2))
cat("\nP(rank = 1)  [probability of being the top-growth economy]:\n")
print(round(colMeans(rank_mat == 1), 3))
cat("\nP(rank <= 2):\n"); print(round(colMeans(rank_mat <= 2), 3))


## Sensitivity — higher volatility & policy shock

In [ ]:
# Scenario A: double Mexico’s volatility
sigma2 <- sigma; sigma2["Mexico"] <- 3.0
growth2 <- matrix(rnorm(n_sims * 3, mean = rep(baseline, each = n_sims),
                        sd = rep(sigma2, each = n_sims)), nrow = n_sims)
rank2 <- t(apply(growth2, 1, function(r) rank(-r, ties.method = "min")))
colnames(rank2) <- names(baseline)
cat("P(rank=1) with higher Mexican volatility:\n")
print(round(colMeans(rank2 == 1), 3))

# Scenario B: +0.8 pp policy boost to Costa Rica
baseline_shock <- baseline; baseline_shock["CostaRica"] <- 4.9
growth3 <- matrix(rnorm(n_sims * 3, mean = rep(baseline_shock, each = n_sims),
                        sd = rep(sigma, each = n_sims)), nrow = n_sims)
rank3 <- t(apply(growth3, 1, function(r) rank(-r, ties.method = "min")))
colnames(rank3) <- names(baseline)
cat("\nP(rank=1) after Costa Rica +0.8 pp shock:\n")
print(round(colMeans(rank3 == 1), 3))


## Bootstrap CI for mean rank of Costa Rica

In [ ]:
cr_ranks <- rank_mat[, "CostaRica"]
set.seed(7)
boot_means <- replicate(2000, mean(sample(cr_ranks, replace = TRUE)))
cat("Mean rank (original):", round(mean(cr_ranks), 2), "\n")
cat("Bootstrap 95% CI    :", round(quantile(boot_means, c(0.025, 0.975)), 2), "\n")


## Interactive Simulation Parameters (ready to tweak)
Change any value and re-execute.


In [ ]:
# === PARAMETERS ===
set.seed(123)
n_sims       <- 1500
baseline     <- c(CostaRica = 4.1, Chile = 2.4, Mexico = 2.0)
sigma        <- c(CostaRica = 1.3, Chile = 1.0, Mexico = 1.5)
policy_shock <- c(CostaRica = 0.0, Chile = 0.0, Mexico = 0.0)  # e.g. c(0.8, 0, 0)

# === ENGINE ===
mu <- baseline + policy_shock
growth_mat <- matrix(rnorm(n_sims * 3, mean = rep(mu, each = n_sims),
                           sd = rep(sigma, each = n_sims)), nrow = n_sims)
colnames(growth_mat) <- names(baseline)
rank_mat <- t(apply(growth_mat, 1, function(r) rank(-r, ties.method = "min")))
colnames(rank_mat) <- names(baseline)

cat("n_sims =", n_sims, "\n")
cat("Mean ranks:\n"); print(round(colMeans(rank_mat), 2))
cat("P(rank = 1):\n"); print(round(colMeans(rank_mat == 1), 3))
cat("P(all ranks <= 2):", round(mean(apply(rank_mat <= 2, 1, all)), 3), "\n")


## Key Results Snapshot (baseline simulation)
| Country    | Baseline growth | Mean simulated rank | P(rank = 1) |
|------------|-----------------|---------------------|-------------|
| CostaRica  | 4.1 %           | ≈ 1.4               | ≈ 0.55–0.60 |
| Chile      | 2.4 %           | ≈ 2.1               | ≈ 0.25      |
| Mexico     | 2.0 %           | ≈ 2.5               | ≈ 0.15–0.20 |

Higher volatility for a country increases the chance it both over- and under-performs; a positive policy shock shifts its rank distribution favourably.
